In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2


In [ ]:
from unsloth import FastLanguageModel
import torch

# ============================
# Base model and configuration
# ============================
model_name    = "unsloth/Qwen3-8B-unsloth-bnb-4bit"
SEED          = 69
MAX_SEQ_LENGTH = 1024

# ============================
# Load model and tokenizer for LoRA finetuning
# ============================
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = model_name,
    max_seq_length  = MAX_SEQ_LENGTH,   # should match your dataset needs
    load_in_4bit    = True,             # required for 4bit LoRA finetuning
    load_in_8bit    = False,
    full_finetuning = False,            # keep this False for LoRA training
)

# ============================
# Prepare model for LoRA / DoRA finetuning
# ============================
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    lora_alpha = 64,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    use_rslora = False,    # keep False unless you explicitly want RS LoRA
    use_dora = True,       # enables DoRA
    loftq_config = None,   # no LoftQ quantization config
)

print("Model loaded and ready for LoRA finetuning.")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from datasets import Dataset
from IPython.display import display

# ---------- Paths ----------
OUTPUT_DIR = "/content/drive/MyDrive/qwen3-8b-unsloth-lora"
OUTPUT_DIR = "/content/drive/MyDrive/qwen3-8b-unsloth-lora-oversampling"



CSV_PATH   = "/content/drive/MyDrive/train.csv"


# ============================
# OVERSAMPLING CONFIG
# ============================
OVERSAMPLING_MODE      = "target_min_count"   # "none" or "target_min_count"
MIN_LABEL_COUNT        = 40                  # target minimum count per label
MAX_DATASET_FACTOR     = 3.0                 # safety: max size = original_size * factor
OVERSAMPLING_SEED      = 42                  # reproducibility

print(f"\nLoading dataset from: {CSV_PATH}")
df = pd.read_csv(CSV_PATH)

# Check required columns exist
required_cols = {"input_finding", "output_disease"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in CSV: {missing}")

# Clean columns
df = df[["input_finding", "output_disease"]].dropna()
df["input_finding"] = df["input_finding"].astype(str).str.strip()
df["output_disease"] = df["output_disease"].astype(str).str.strip()

print("\nNumber of rows after filtering:", len(df))
print("Columns:", list(df.columns))
display(df.head())

# ============================
# Helper: label counts
# ============================
def compute_label_counts(df_in):
    exploded = (
        df_in["output_disease"]
        .str.split(",")
        .explode()
        .str.strip()
    )
    exploded = exploded[exploded != ""]
    return exploded.value_counts()

# ============================
# Label list and counts BEFORE
# ============================
label_counts_before = compute_label_counts(df)
all_labels = sorted(label_counts_before.index.tolist())
allowed_labels = ", ".join(all_labels)

print("\nTotal unique labels:", len(all_labels))
print("Label preview:", allowed_labels[:200], "...\n")

print("Top 10 most frequent labels BEFORE oversampling:")
print(label_counts_before.head(10))

# ============================
# Plot with shared Y axis
# ============================
def plot_label_distributions(counts_before, counts_after):
    ymax = max(counts_before.max(), counts_after.max())

    # Before
    plt.figure(figsize=(12, 6))
    sorted_before = counts_before.sort_values(ascending=False)
    plt.bar(range(len(sorted_before)), sorted_before.values)
    plt.xticks(range(len(sorted_before)), sorted_before.index, rotation=90)
    plt.title("Label Distribution BEFORE Oversampling")
    plt.xlabel("Disease Label")
    plt.ylabel("Frequency")
    plt.ylim(0, ymax)
    plt.tight_layout()
    plt.show()

    # After
    plt.figure(figsize=(12, 6))
    sorted_after = counts_after.sort_values(ascending=False)
    plt.bar(range(len(sorted_after)), sorted_after.values)
    plt.xticks(range(len(sorted_after)), sorted_after.index, rotation=90)
    plt.title("Label Distribution AFTER Oversampling")
    plt.xlabel("Disease Label")
    plt.ylabel("Frequency")
    plt.ylim(0, ymax)
    plt.tight_layout()
    plt.show()

orig_n = len(df)

# ============================
# OVERSAMPLING: target_min_count with minority only rows
# ============================
if OVERSAMPLING_MODE == "none":
    print("\nOversampling mode: none (using original dataset).")
    df_balanced = df.copy()

elif OVERSAMPLING_MODE == "target_min_count":
    print("\nOversampling mode: target_min_count")
    print(f"Target minimum count per label: {MIN_LABEL_COUNT}")

    rng = np.random.default_rng(OVERSAMPLING_SEED)

    df_balanced = df.copy()
    label_counts = label_counts_before.copy()

    # Identify minority labels
    minority_labels = label_counts[label_counts < MIN_LABEL_COUNT].index.tolist()
    minority_set = set(minority_labels)

    print(f"\nNumber of minority labels (< {MIN_LABEL_COUNT}): {len(minority_labels)}")

    # Precompute, for each row, the set of labels and whether all of them are minority
    def row_labels_set(s):
        return {l.strip() for l in str(s).split(",") if l.strip() != ""}

    df["label_set"] = df["output_disease"].apply(row_labels_set)
    df["all_labels_minority"] = df["label_set"].apply(
        lambda labset: len(labset) > 0 and labset.issubset(minority_set)
    )

    extra_indices = []

    for label in minority_labels:
        count = label_counts[label]
        if count >= MIN_LABEL_COUNT:
            continue

        deficit = MIN_LABEL_COUNT - count

        # Rows that contain this label
        mask_label = df["output_disease"].str.contains(
            rf"(?:^|,\s*){re.escape(label)}(?:,|$)"
        )
        # Only rows where all labels are minority
        mask_rare_only = df["all_labels_minority"]

        candidate_indices = df[mask_label & mask_rare_only].index.to_list()

        if not candidate_indices:
            print(
                f"Label '{label}' count {count}, but no rare-only rows found. "
                "Skipping oversampling for this label to avoid boosting majority classes."
            )
            continue

        sampled = rng.choice(candidate_indices, size=deficit, replace=True)
        extra_indices.extend(sampled)

        print(f"Label '{label}': {count} -> {count + deficit} (oversampled {deficit} rows)")

    if extra_indices:
        df_extra = df.loc[extra_indices].drop(columns=["label_set", "all_labels_minority"])
        df_balanced = pd.concat([df_balanced, df_extra], ignore_index=True)
    else:
        df_balanced = df_balanced.drop(columns=["label_set", "all_labels_minority"])

    # Safety check
    max_allowed = int(orig_n * MAX_DATASET_FACTOR)
    if len(df_balanced) > max_allowed:
        print(
            f"\nWarning: oversampled dataset size {len(df_balanced)} "
            f"exceeds MAX_DATASET_FACTOR={MAX_DATASET_FACTOR} * {orig_n}={max_allowed}. "
            "You may want to lower MIN_LABEL_COUNT or MAX_DATASET_FACTOR."
        )

    print(f"\nOriginal dataset size: {orig_n}")
    print(f"New dataset size:       {len(df_balanced)}")

else:
    raise ValueError(f"Unknown OVERSAMPLING_MODE: {OVERSAMPLING_MODE}")

# Clean helper columns if they exist
for col in ["label_set", "all_labels_minority"]:
    if col in df_balanced.columns:
        df_balanced = df_balanced.drop(columns=[col])

# ============================
# COUNTS AFTER AND PLOTS
# ============================
label_counts_after = compute_label_counts(df_balanced)

print("\nLabel frequency summary AFTER oversampling (top 10):")
print(label_counts_after.head(10))

print("\nPlotting label distributions BEFORE and AFTER oversampling...")
plot_label_distributions(label_counts_before, label_counts_after)

# ============================
# CONVERT TO HF DATASET
# ============================
hf_dataset = Dataset.from_pandas(df_balanced, preserve_index=False)
print("\nFinal HF Dataset:")
print(hf_dataset)


In [ ]:
# import numpy as np

# # System prompt used in every training example
# system_prompt = (
#     "You are a clinical Named Entity Recognition (NER) and multi-label classification model. "
#     "Read the abdominal radiology findings and identify all diseases that are present. "
#     "Use only disease names from the allowed disease label list. "
#     "Return the diseases as a comma separated list using the exact wording from the list. "
#     "If none apply, output: None. "
#     f"Allowed disease label list: {allowed_labels}."
# )

# def format_batch(batch):
#     texts = []
#     for finding, labels in zip(batch["input_finding"], batch["output_disease"]):
#         finding = str(finding).strip()
#         labels = str(labels).strip()

#         messages = [
#             {"role": "system", "content": system_prompt},
#             {
#                 "role": "user",
#                 "content": (
#                     "Clinical findings:\n"
#                     f"{finding}\n\n"
#                     "List all diseases present using only labels from the allowed disease list. "
#                     "Separate multiple diseases with commas. If none apply, output: None."
#                 ),
#             },
#             {"role": "assistant", "content": labels},
#         ]

#         # Convert chat messages to plain training text
#         text = tokenizer.apply_chat_template(
#             messages,
#             tokenize=False,
#             add_generation_prompt=False,
#             enable_thinking=False,
#         )
#         texts.append(text)

#     return {"text": texts}

# print("Applying chat template to the dataset...")
# processed_dataset = hf_dataset.map(
#     format_batch,
#     batched=True,
#     remove_columns=hf_dataset.column_names,
# )

# print("Example processed text:")
# example_text = processed_dataset[0]["text"]
# print(example_text)

# # -------------------------------
# # Compute token length statistics
# # -------------------------------
# lengths = []

# for sample in processed_dataset["text"]:
#     encoded = tokenizer(
#         sample,
#         add_special_tokens=False,
#     )
#     lengths.append(len(encoded["input_ids"]))

# lengths = np.array(lengths)

# print("\nToken length statistics for training texts:")
# print("Number of samples:", len(lengths))
# print("Min length:", int(lengths.min()))
# print("Max length:", int(lengths.max()))
# print("Mean length:", float(lengths.mean()))
# print("Median length (50th percentile):", int(np.percentile(lengths, 50)))
# print("90th percentile:", int(np.percentile(lengths, 90)))
# print("95th percentile:", int(np.percentile(lengths, 95)))
# print("99th percentile:", int(np.percentile(lengths, 99)))

# example_len = len(tokenizer(example_text, add_special_tokens=False)["input_ids"])
# print("\nToken length of example 0:", example_len)


In [ ]:
# Version B

import numpy as np

# ----------------------------------------------------
# PROMPT VARIATION CONFIG
# ----------------------------------------------------
USE_PROMPT_VARIATION = True   # set False to disable augmentation

# Multiple system prompt variants
system_prompt_variants = [
    (
        "You are a clinical Named Entity Recognition (NER) and multi-label classification model. "
        "Read the abdominal radiology findings and identify all diseases that are present. "
        "Use only disease names from the allowed disease label list. "
        "Return the diseases as a comma separated list using the exact wording from the list. "
        "If none apply, output: No acute abnormality"
        f"Allowed disease label list: {allowed_labels}."
    ),
    (
        "You are a clinical NER and multi-disease extraction model. Your task is to examine "
        "abdominal radiology findings and output all diseases that appear in the text. "
        "Use only the disease names exactly as they appear in the allowed disease label list. "
        "Return results as a comma-separated list, or output: No acute abnormality, if no diseases apply. "
        f"The allowed disease label list is: {allowed_labels}."
    ),
    (
        "You are an expert clinical NLP system specialized in disease extraction. Your job is to "
        "identify every disease present in abdominal radiology findings, selecting only from the "
        "allowed disease label list. Output a comma-separated list of labels, or No acute abnormality if no labels "
        "match. "
        f"Allowed disease label list: {allowed_labels}."
    ),
]

# Multiple user prompt variants
user_prompt_variants = [
    (
        "Clinical findings:\n{finding}\n\n"
        "List all diseases present using only labels from the allowed disease list. "
        "Use commas to separate diseases. If none apply, output: None."
    ),
    (
        "Below are the clinical findings:\n{finding}\n\n"
        "Identify all diseases mentioned using ONLY the allowed disease label list. "
        "Provide a comma-separated list, or None if no disease is present."
    ),
    (
        "Given the following abdominal CT findings:\n{finding}\n\n"
        "Extract every disease present using only valid labels from the allowed list. "
        "Return them as a comma-separated list, or None if no labels apply."
    ),
]

# ----------------------------------------------------
# FORMAT BATCH WITH RANDOM PROMPT VARIATION
# ----------------------------------------------------
def format_batch(batch):
    texts = []
    for finding, labels in zip(batch["input_finding"], batch["output_disease"]):
        finding = str(finding).strip()
        labels = str(labels).strip()

        # choose variant
        if USE_PROMPT_VARIATION:
            system_prompt = np.random.choice(system_prompt_variants)
            user_template = np.random.choice(user_prompt_variants)
        else:
            # original (no augmentation)
            system_prompt = system_prompt_variants[0]
            user_template = user_prompt_variants[0]

        user_prompt = user_template.format(finding=finding)

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": labels},
        ]

        # Convert chat messages to plain training text
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )
        texts.append(text)

    return {"text": texts}

print("Applying chat template to the dataset...")
processed_dataset = hf_dataset.map(
    format_batch,
    batched=True,
    remove_columns=hf_dataset.column_names,
)

print("Example processed text:")
example_text = processed_dataset[0]["text"]
print(example_text)

# ----------------------------------------------------
# TOKEN LENGTH STATISTICS
# ----------------------------------------------------
lengths = []

for sample in processed_dataset["text"]:
    encoded = tokenizer(
        sample,
        add_special_tokens=False,
    )
    lengths.append(len(encoded["input_ids"]))

lengths = np.array(lengths)

print("\nToken length statistics for training texts:")
print("Number of samples:", len(lengths))
print("Min length:", int(lengths.min()))
print("Max length:", int(lengths.max()))
print("Mean length:", float(lengths.mean()))
print("Median length (50th percentile):", int(np.percentile(lengths, 50)))
print("90th percentile:", int(np.percentile(lengths, 90)))
print("95th percentile:", int(np.percentile(lengths, 95)))
print("99th percentile:", int(np.percentile(lengths, 99)))

example_len = len(tokenizer(example_text, add_special_tokens=False)["input_ids"])
print("\nToken length of example 0:", example_len)


In [ ]:
from trl import SFTTrainer, SFTConfig

# Training hyperparameters
BATCH_SIZE     = 2           # Works on most 16–24 GB GPUs with Qwen3-8B 4bit LoRA
GRAD_ACCUM     = 4           # Effective batch size = 2 × 4 = 8 (good stability, low VRAM)
EPOCHS         = 3           # Dataset is small (1236 samples); 3 epochs is ideal

# LR             = 2e-4        # Standard LoRA LR for 8B models; adjust lower if loss becomes unstable
LR             = 1e-4        # Standard LoRA LR for 8B models; adjust lower if loss becomes unstable

print("Setting up training configuration...")

# Prefer bf16 when supported (more stable, faster).
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = SFTConfig(

    # Max tokens per sequence. Covers all samples (max ≈ 1039 tokens),
    # while being friendly to Qwen3-8B VRAM footprint.
    max_seq_length = MAX_SEQ_LENGTH,

    # Column name containing full training text after apply_chat_template.
    dataset_text_field = "text",

    # Real batch size per device/GPU. Qwen3-8B 4bit LoRA typically supports 1–2.
    per_device_train_batch_size = BATCH_SIZE,

    # Accumulate gradients to simulate larger effective batch without more memory.
    gradient_accumulation_steps = GRAD_ACCUM,

    # Number of full passes through the dataset. Works well with 1236 samples.
    num_train_epochs = EPOCHS,

    # Learning rate for LoRA adapter weights. 2e-4 is aggressive but stable for 4bit LoRA.
    learning_rate = LR,

    # Warmup steps prevent spikes in early training.
    warmup_steps = 50,

    # Logs progress every 10 steps.
    logging_steps = 10,

    # Saves a checkpoint at the end of each epoch.
    save_strategy = "epoch",

    # Output folder for checkpoints, logs, and final LoRA weights.
    output_dir = OUTPUT_DIR,

    # Use memory-efficient 8-bit AdamW optimizer (important for 8B model).
    optim = "adamw_8bit",

    # Whether to use bf16. Preferred when GPU supports it (A100, 4090, etc).
    bf16 = use_bf16,

    # Fallback to fp16 when bf16 is not supported.
    fp16 = not use_bf16,

    # Random seed for reproducible training.
    seed = SEED,
)

trainer = SFTTrainer(
    model         = model,              # Qwen3-8B 4bit LoRA-ready model
    tokenizer     = tokenizer,          # Must match model
    train_dataset = processed_dataset,  # Your processed dataset with template-applied text
    args          = training_args,
    packing       = True,               # Better GPU utilization when samples are shorter than max length
)

print("Trainer is ready.")


In [ ]:
import os

# Start training loop
print("Starting training...")
trainer.train()
print("Training completed.")

print("Saving fine tuned model and tokenizer to:", OUTPUT_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save LoRA adapter weights and config
model.save_pretrained(OUTPUT_DIR)

# Save tokenizer with the chat template and special tokens
tokenizer.save_pretrained(OUTPUT_DIR)

print("Done.")
